# GPCR リガンド親和性予測
## ProtBERT 埋め込み + 機械学習で G蛋白連関受容体とペプチドリガンドの相互作用を予測

## 実習概要

本ノートブックでは、以下を学習します:

1. **タンパク質-タンパク質相互作用(PPI)の予測**
   - 受容体(GPCR)とリガンド(ペプチド)の配列から相互作用を予測
   - シーケンス情報だけで物理的な相互作用をモデル化

2. **GPCR について**
   - 細胞膜に7回貫通する受容体ファミリー
   - ホルモン、神経伝達物質、光などのシグナルを感知
   - 約35%の医薬品のターゲット

3. **ペプチドリガンド**
   - 短いアミノ酸配列(通常10～50残基)
   - ホルモンやニューロペプチドとして機能

4. **予測モデルの構築**
   - GPCR とペプチドを ProtBERT で埋め込み
   - 両者の相互作用特徴を抽出
   - 機械学習で二値分類(結合 vs 非結合)

## 0. 必要ライブラリのインストール

In [ ]:
# 初回のみ実行
# !pip install transformers torch scikit-learn xgboost pandas numpy matplotlib seaborn scipy

## 1. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score
)
from xgboost import XGBClassifier
import torch
from transformers import AutoTokenizer, AutoModel
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
np.random.seed(42)

print("✓ ライブラリのインポート完了")

## 2. ProtBERT モデルのロード

In [ ]:
print("ProtBERT をロード中...")

model_name = "Rostlab/prot_bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"✓ ProtBERT ロード完了")
print(f"  デバイス: {device}")

## 3. 埋め込み関数の定義

In [ ]:
def get_protein_embedding(sequence, tokenizer, model, device):
    """
    アミノ酸配列をProtBERT埋め込みに変換
    """
    sequence_spaced = " ".join(sequence)
    inputs = tokenizer(sequence_spaced, return_tensors="pt", max_length=4096,
                      truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

print("✓ 埋め込み関数の定義完了")

## 4. データセットの構築

In [ ]:
# GPCR とネイティブリガンド
gpcr_ligand_pairs = [
    {
        "gpcr_name": "Beta-2 adrenergic receptor",
        "gpcr_seq": "MGQPGNGSAFLLAPNGSGLQWPHIIGQQPNGLSIGFQNGTSDVTAAYVQQVHNGVRWLF",
        "ligand_name": "Epinephrine",
        "ligand_seq": "CNHCOH",
        "label": 1
    },
    {
        "gpcr_name": "Glucagon receptor",
        "gpcr_seq": "MASGPRGPPAPGWQASGPAGCAGPASAAQRAAAQE",
        "ligand_name": "Glucagon",
        "ligand_seq": "HSQGAARGGLGRG",
        "label": 1
    },
    {
        "gpcr_name": "GLP-1 receptor",
        "gpcr_seq": "MRVPLSVALFGLVGSLQTRSNVNKRRN",
        "ligand_name": "GLP-1",
        "ligand_seq": "HAEGTFTSDVSSYELEG",
        "label": 1
    },
    {
        "gpcr_name": "Muscarinic receptor M1",
        "gpcr_seq": "MNNGNNQSVHCPDPMGGLSPGE",
        "ligand_name": "Acetylcholine",
        "ligand_seq": "ACCG",
        "label": 1
    },
    {
        "gpcr_name": "Serotonin receptor 2A",
        "gpcr_seq": "MLKCLAVTLAVCLSNALVVLSE",
        "ligand_name": "Serotonin",
        "ligand_seq": "NCCGN",
        "label": 1
    },
    {
        "gpcr_name": "Opsin (Rhodopsin)",
        "gpcr_seq": "MNGTEGPNFYVPFSNATGVVRSPFEYPQYYLAEPWQFSMLAAYMFLLIVLGFPINFLTLYVTVQHKKLR",
        "ligand_name": "Retinal",
        "ligand_seq": "CRNRN",
        "label": 1
    },
]

# デコイペプチド
decoy_peptides = [
    "MVLSPADKTNVK",
    "DIQMTQSPSSLSA",
    "GIVEQCCTSICSLY",
    "MNLGHQSLGTLMA",
    "KKPKLLYCSASQ",
    "EGFRGKPHDK",
    "AVKVHQ",
    "QTPRW",
    "SDRGHP",
    "MKNVLPLP",
    "DPVQWWVN",
    "GKFLNGPYA",
]

# ポジティブペア
data = gpcr_ligand_pairs.copy()

# ネガティブペア
for gpcr in gpcr_ligand_pairs:
    for decoy in decoy_peptides:
        data.append({
            "gpcr_name": gpcr["gpcr_name"],
            "gpcr_seq": gpcr["gpcr_seq"],
            "ligand_name": "Decoy",
            "ligand_seq": decoy,
            "label": 0
        })

df_raw = pd.DataFrame(data)

print(f"✓ データセット構築完了")
print(f"  総サンプル数: {len(df_raw)}")
print(f"  ポジティブペア(結合): {(df_raw['label']==1).sum()}")
print(f"  ネガティブペア(非結合): {(df_raw['label']==0).sum()}")

## 5. GPCR の埋め込み計算

In [ ]:
print("GPCR 埋め込みを計算中...")
gpcr_embeddings = {}

for gpcr_name in df_raw['gpcr_name'].unique():
    gpcr_seq = df_raw[df_raw['gpcr_name'] == gpcr_name]['gpcr_seq'].iloc[0]
    try:
        embedding = get_protein_embedding(gpcr_seq, tokenizer, model, device)
        gpcr_embeddings[gpcr_name] = embedding
        print(f"  ✓ {gpcr_name[:35]:<35} : {len(gpcr_seq):>3} aa")
    except Exception as e:
        print(f"  ✗ エラー: {gpcr_name}")
        gpcr_embeddings[gpcr_name] = np.zeros(1280)

print(f"✓ GPCR 埋め込み完了: {len(gpcr_embeddings)} 種類")

## 6. ペプチドの埋め込み計算

In [ ]:
print("ペプチド埋め込みを計算中...")
peptide_embeddings = {}

unique_peptides = df_raw['ligand_seq'].unique()
for idx, peptide_seq in enumerate(unique_peptides):
    try:
        embedding = get_protein_embedding(peptide_seq, tokenizer, model, device)
        peptide_embeddings[peptide_seq] = embedding
        if (idx + 1) % 5 == 0:
            print(f"  {idx+1}/{len(unique_peptides)} 完了", end="\r")
    except Exception as e:
        print(f"エラー: {peptide_seq}")
        peptide_embeddings[peptide_seq] = np.zeros(1280)

print(f"\n✓ ペプチド埋め込み完了: {len(peptide_embeddings)} 種類")

## 7. 相互作用特徴の抽出

In [ ]:
def extract_interaction_features(gpcr_emb, peptide_emb, method="concat"):
    """
    GPCR とペプチド埋め込みから相互作用特徴を抽出
    """
    if method == "concat":
        return np.concatenate([gpcr_emb, peptide_emb])
    elif method == "diff":
        return np.abs(gpcr_emb - peptide_emb)
    elif method == "hadamard":
        return gpcr_emb * peptide_emb
    else:
        return np.concatenate([gpcr_emb, peptide_emb])

print("相互作用特徴を抽出中...")

features_list = []
for idx, row in df_raw.iterrows():
    gpcr_emb = gpcr_embeddings[row['gpcr_name']]
    peptide_emb = peptide_embeddings[row['ligand_seq']]
    features = extract_interaction_features(gpcr_emb, peptide_emb, method="concat")
    features_list.append(features)
    
    if (idx + 1) % 20 == 0:
        print(f"  {idx+1}/{len(df_raw)} 完了", end="\r")

X = np.array(features_list)
y = df_raw['label'].values

print(f"\n✓ 特徴量抽出完了")
print(f"  入力サンプル数: {X.shape[0]}")
print(f"  特徴量次元: {X.shape[1]}")

## 8. データ分割

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ データ分割完了")
print(f"  訓練: {len(X_train)}")
print(f"  テスト: {len(X_test)}")

## 9. ランダムフォレストで分類

In [ ]:
print("ランダムフォレストを訓練中...")

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print(f"\n✓ ランダムフォレストの結果")
print(f"  訓練精度: {rf.score(X_train, y_train):.3f}")
print(f"  テスト精度: {rf.score(X_test, y_test):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.3f}")
print(f"  F1スコア: {f1_score(y_test, y_pred_rf):.3f}")

print(f"\n分類レポート:")
print(classification_report(y_test, y_pred_rf,
                          target_names=["ネガティブ", "ポジティブ"]))

## 10. XGBoost で分類

In [ ]:
print("XGBoost を訓練中...")

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_pred_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print(f"\n✓ XGBoost の結果")
print(f"  訓練精度: {xgb.score(X_train, y_train):.3f}")
print(f"  テスト精度: {xgb.score(X_test, y_test):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.3f}")
print(f"  F1スコア: {f1_score(y_test, y_pred_xgb):.3f}")

print(f"\n分類レポート:")
print(classification_report(y_test, y_pred_xgb,
                          target_names=["ネガティブ", "ポジティブ"]))

## 11. モデル比較

In [ ]:
comparison_df = pd.DataFrame({
    "モデル": ["ランダムフォレスト", "XGBoost"],
    "テスト精度": [
        rf.score(X_test, y_test),
        xgb.score(X_test, y_test)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_pred_proba_rf),
        roc_auc_score(y_test, y_pred_proba_xgb)
    ],
    "F1スコア": [
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ]
})

print("\n📊 モデル比較")
print(comparison_df.to_string(index=False))

# ROC曲線
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC曲線
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb)

axes[0].plot(fpr_rf, tpr_rf, label=f"RF (AUC={roc_auc_score(y_test, y_pred_proba_rf):.3f})", linewidth=2)
axes[0].plot(fpr_xgb, tpr_xgb, label=f"XGB (AUC={roc_auc_score(y_test, y_pred_proba_xgb):.3f})", linewidth=2)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("偽陽性率")
axes[0].set_ylabel("真陽性率")
axes[0].set_title("ROC 曲線")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_pred_proba_rf)
precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_pred_proba_xgb)

axes[1].plot(recall_rf, precision_rf, label="ランダムフォレスト", linewidth=2)
axes[1].plot(recall_xgb, precision_xgb, label="XGBoost", linewidth=2)
axes[1].set_xlabel("再現率")
axes[1].set_ylabel("適合率")
axes[1].set_title("Precision-Recall 曲線")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. 混同行列

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["ネガティブ", "ポジティブ"],
            yticklabels=["ネガティブ", "ポジティブ"])
axes[0].set_title("ランダムフォレスト")
axes[0].set_ylabel("実際")
axes[0].set_xlabel("予測")

sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["ネガティブ", "ポジティブ"],
            yticklabels=["ネガティブ", "ポジティブ"])
axes[1].set_title("XGBoost")
axes[1].set_ylabel("実際")
axes[1].set_xlabel("予測")

plt.tight_layout()
plt.show()

## 13. 新しいペアでの予測

In [ ]:
test_pairs = [
    {
        "gpcr_name": "Oxytocin receptor",
        "gpcr_seq": "MNNNCAGFLVGVYLVGVLGDQDVEGELG",
        "peptide_name": "Oxytocin (native)",
        "peptide_seq": "CYIQNCPLG",
    },
    {
        "gpcr_name": "Oxytocin receptor",
        "gpcr_seq": "MNNNCAGFLVGVYLVGVLGDQDVEGELG",
        "peptide_name": "Random peptide",
        "peptide_seq": "KKKKPPPQQQ",
    },
]

print("\n🔍 新しいペアでの予測\n")

for pair in test_pairs:
    try:
        gpcr_emb = get_protein_embedding(pair["gpcr_seq"], tokenizer, model, device)
        peptide_emb = get_protein_embedding(pair["peptide_seq"], tokenizer, model, device)
        features = extract_interaction_features(gpcr_emb, peptide_emb, method="concat")
        
        pred_rf = rf.predict([features])[0]
        prob_rf = rf.predict_proba([features])[0]
        
        pred_xgb = xgb.predict([features])[0]
        prob_xgb = xgb.predict_proba([features])[0]
        
        pred_label_rf = "結合" if pred_rf == 1 else "非結合"
        pred_label_xgb = "結合" if pred_xgb == 1 else "非結合"
        
        print(f"GPCR: {pair['gpcr_name']}")
        print(f"ペプチド: {pair['peptide_name']}")
        print(f"  RF: {pred_label_rf} (確信度: {max(prob_rf):.1%})")
        print(f"  XGB: {pred_label_xgb} (確信度: {max(prob_xgb):.1%})")
        print()
    except Exception as e:
        print(f"エラー: {e}")
        print()

## 14. 実習のまとめ

In [ ]:
print("\n" + "="*70)
print("\t実習のまとめ")
print("="*70)

print(f"""
1️⃣ タンパク質-タンパク質相互作用(PPI)の予測:
   - GPCR とペプチドリガンドの相互作用を、
     配列情報だけから予測できた
   - ProtBERT が構造・機能情報を自動学習

2️⃣ 特徴抽出の工夫:
   - GPCR埋め込み + ペプチド埋め込みの連結
   - 2つのタンパク質の『組み合わせ』として相互作用を表現

3️⃣ クラス不均衡への対処:
   - ネガティブペア >> ポジティブペア
   - XGBoost の scale_pos_weight で対応
   - F1スコアが重要な評価指標

4️⃣ 実務への応用:
   - 創薬スクリーニング: 有望なリガンド候補の抽出
   - ゲノム解析: 遺伝子変異が親和性に与える影響予測
   - ペプチドデザイン: より強く結合するペプチドの設計
""")

print("="*70)

## 15. 発展課題

### 課題 1: より大規模なデータセット
BindingDB, PDBbind, IUPHAR データベースから実データを取得

### 課題 2: 結合親和性の回帰予測
2値分類ではなく、Kd や IC50 などの実験値を予測

### 課題 3: 構造情報の統合
AlphaFold で予測した3D構造から幾何学的特徴を抽出

### 課題 4: 他の言語モデルの比較
ESM-2, ProtT5 など他のモデルで精度を比較

### 課題 5: 特徴量のアブレーション
concat, diff, hadamard など複数の組み合わせ方法を比較